In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option('display.max_rows', None)

In [80]:
X_train = pd.read_csv('csv/final_datasets/for_linear_model/MF/predictors_train_filtered.csv')
X_test = pd.read_csv('csv/final_datasets/for_linear_model/MF/predictors_test_filtered.csv')
y_train = pd.read_csv('csv/final_datasets/for_linear_model/MF/target_train_filtered_logariphmed.csv')
y_test = pd.read_csv('csv/final_datasets/for_linear_model/MF/target_test_filtered_logariphmed.csv')

In [4]:
train_players = pd.read_csv('csv/final_datasets/for_linear_model/MF/train_players_names.csv')
test_players = pd.read_csv('csv/final_datasets/for_linear_model/MF/test_players_names.csv')

In [5]:
all_cols = X_train.columns
all_cols

Index(['Age', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK',
       'PKatt', 'CrdY', 'CrdR', 'Gls.1', 'Ast.1', 'G+A.1', 'G-PK.1', 'G+A-PK',
       '2CrdY', 'Fls', 'Fld', 'Off', 'Crs', 'Int', 'TklW', 'OG', 'Sh', 'SoT',
       'SoT%', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1'],
      dtype='object')

## Регрессор

In [83]:
LR_regressor = LinearRegression()
LR_regressor.fit(X_train, y_train)

LinearRegression()

## Предсказанные значения

In [84]:
y_pred_log = LR_regressor.predict(X_test)
y_pred = np.exp(y_pred_log)
y_test_original = np.exp(y_test)

y_train_pred = np.exp(LR_regressor.predict(X_train))
y_train_original = np.exp(y_train)

## Метрики

In [85]:
mae_test = mean_absolute_error(y_test_original, y_pred)
rmse_test = np.sqrt(mean_squared_error(y_test_original, y_pred))
r2_test = r2_score(y_test_original, y_pred)
mape_test = mean_absolute_percentage_error(y_test_original, y_pred)

mae_train = mean_absolute_error(y_train_original, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train_original, y_train_pred))
r2_train = r2_score(y_train_original, y_train_pred)
mape_train = mean_absolute_percentage_error(y_train_original, y_train_pred)

metrics = pd.DataFrame({
    'MAE':[mae_test, mae_train],
    'RMSE':[rmse_test, rmse_train],
    'MAPE':[mape_test, mape_train],
    'R2':[r2_test, r2_train]
}, index = ['test', 'train'])

## Остатки

In [86]:
y_pred = pd.Series(np.reshape(y_pred, len(y_pred)), name='Value_pred')
y_train_pred = pd.Series(np.reshape(y_train_pred, len(y_train_pred)), name='Value_pred')

comp_leftovers_test = pd.concat([test_players, y_pred, y_test_original], axis=1)
comp_leftovers_train = pd.concat([train_players, y_train_pred, y_train_original], axis=1)

## Коэффициенты

In [87]:
coeffs = LR_regressor.coef_[0]
coeffs_df = pd.DataFrame({
    'coeff':coeffs.astype('float')
}, index=X_train.columns)
coeffs_df = coeffs_df.sort_values(by='coeff', key=abs, ascending=False)

<b>Все переменные</b>

In [12]:
coeffs_df

,coeff
90s,16.319206
Min,-16.089723
league_GB1,0.567529
Gls.1,0.465130
Age,-0.421172
G+A-PK,-0.377476
league_IT1,0.365412
league_ES1,0.299258
league_L1,0.278807
Ast,0.243098


In [13]:
metrics

,MAE,RMSE,MAPE,R2
test,5.774406,15.347800,1.022365,0.478862
train,5.387281,11.309522,0.841682,0.440134


In [14]:
comp_leftovers_test

,Player,Value_pred,Value
0,Sandi Lovric,14.820001,8.500
1,Danila Godiaev,0.937032,1.500
2,Inigo Ruiz de Galarreta,3.693768,3.000
3,Johnny Cardoso,13.330161,25.000
4,Unai Lopez,4.158035,2.500
5,Florian Grillitsch,2.577003,2.000
6,Mahmoud Dahoud,2.684127,3.500
7,Teji Savanier,7.063929,2.500
8,Djaoui Cisse,6.261746,8.000
9,Dean Huiberts,0.863534,0.300


In [15]:
comp_leftovers_train

,Player,Value_pred,Value
0,Mateo Flores,2.051606,1.00
1,Lennard Hens,1.174727,0.40
2,Orkun Kokcu,11.187435,30.00
3,Ylber Ramadani,4.929461,1.80
4,Aleksandr Kovalenko,1.257167,1.50
5,Hidemasa Morita,1.807709,13.00
6,Felix Nmecha,12.075562,28.00
7,Philipp Sander,4.495042,4.00
8,Kacper Urbanski,5.158894,6.00
9,Jordan Veretout,2.622854,4.00


In [16]:
comp_leftovers_train[comp_leftovers_train['Player'].str.contains('Batxi')]

,Player,Value_pred,Value
536,Batxi,6.561134,5.0


<b>Потенциальные переменные для добавления</b>

In [79]:
base_cols = ['G+A', 'Gls.1', 'SoT/90', '90s', 'Fld', 'CrdY', 'SoT%', 'Age', 'Crs', 'Int', 'PK', 'Sh', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']

In [46]:
cols_to_add = [col for col in all_cols if not(col in base_cols)]
new_cols = pd.DataFrame({
    'mae':[mae_test],
    'mape':[mape_test],
    'r2':[r2_test]
}, index=['base'])
for col in cols_to_add:
    X_train_new = X_train[base_cols + [col]]
    X_test_new = X_test[base_cols + [col]]
    LR_regressor_new = LinearRegression()
    LR_regressor_new.fit(X_train_new, y_train)
    y_pred_new = np.exp(LR_regressor_new.predict(X_test_new))
    mae_new = mean_absolute_error(y_pred_new, y_test_original)
    mape_new = mean_absolute_percentage_error(y_pred_new, y_test_original)
    r2_new = r2_score(y_pred_new, y_test_original)
    if mae_new<mae_test or mape_new<mape_test or r2_new > r2_test:
        row = pd.DataFrame([[mae_new, mape_new, r2_new]], columns=new_cols.columns, index=[col])
        new_cols = pd.concat([new_cols, row])
new_cols

,mae,mape,r2
base,5.686543,0.951272,0.572145
Min,5.652037,1.007650,-0.030386
PK,5.530911,1.014825,0.047364
PKatt,5.550080,1.022319,0.009378
2CrdY,5.684844,1.008763,0.030107
Off,5.598393,1.004720,-0.068392
Int,5.653316,1.010213,0.051427
TklW,5.660771,1.010023,0.027543
Sh,5.679193,1.009237,0.036855
G/Sh,5.678686,1.008258,0.024269


<b>Потенциальные переменные для удаления</b>

In [81]:
cols_to_delete = [col for col in base_cols if not(col.startswith('league'))]
del_cols = pd.DataFrame({
    'mae':[mae_test],
    'mape':[mape_test],
    'r2':[r2_test]
}, index=['base'])
for col in cols_to_delete:
    X_train_new = X_train[[c for c in base_cols if c != col]]
    X_test_new = X_test[[c for c in base_cols if c != col]]
    LR_regressor_new = LinearRegression()
    LR_regressor_new.fit(X_train_new, y_train)
    y_pred_new = np.exp(LR_regressor_new.predict(X_test_new))
    mae_new = mean_absolute_error(y_pred_new, y_test_original)
    mape_new = mean_absolute_percentage_error(y_pred_new, y_test_original)
    r2_new = r2_score(y_pred_new, y_test_original)
    if mae_new < mae_test or mape_new < mape_test or r2_new > r2_test:
        row = pd.DataFrame([[mae_new, mape_new, r2_new]], columns=del_cols.columns, index=[col])
        del_cols = pd.concat([del_cols, row])
del_cols

,mae,mape,r2
base,5.400191,0.931033,0.642989
SoT/90,5.367201,1.034970,0.237663
90s,5.254602,1.060704,0.521605
Sh,5.369418,1.036733,0.295511


<b>Модель 1</b>

In [17]:
X_train = X_train[['G+A', 'G+A.1', 'SoT', 'Ast.1', 'Gls.1', 'SoT/90', '90s', 'Fld', 'PK', 'CrdY', 'SoT%', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G+A', 'G+A.1', 'SoT', 'Ast.1', 'Gls.1', 'SoT/90', '90s', 'Fld', 'PK', 'CrdY', 'SoT%', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [23]:
coeffs_df

,coeff
league_GB1,0.556580
90s,0.520633
Age,-0.433309
league_IT1,0.363924
league_ES1,0.283295
G+A,0.282728
Ast.1,0.282216
league_L1,0.274195
Gls.1,0.270752
league_FR1,0.238401


In [24]:
metrics

,MAE,RMSE,MAPE,R2
test,5.597816,14.268647,0.945403,0.549571
train,5.390391,11.065701,0.862653,0.464014


In [25]:
comp_leftovers_test

,Player,Value_pred,Value
0,Sandi Lovric,10.773762,8.500
1,Danila Godiaev,0.919997,1.500
2,Inigo Ruiz de Galarreta,3.228167,3.000
3,Johnny Cardoso,11.559130,25.000
4,Unai Lopez,3.952923,2.500
5,Florian Grillitsch,2.380475,2.000
6,Mahmoud Dahoud,2.963600,3.500
7,Teji Savanier,6.948817,2.500
8,Djaoui Cisse,5.490980,8.000
9,Dean Huiberts,0.905834,0.300


In [26]:
comp_leftovers_train

,Player,Value_pred,Value
0,Mateo Flores,2.204801,1.00
1,Lennard Hens,1.158568,0.40
2,Orkun Kokcu,10.415220,30.00
3,Ylber Ramadani,4.376401,1.80
4,Aleksandr Kovalenko,1.374605,1.50
5,Hidemasa Morita,1.553983,13.00
6,Felix Nmecha,10.264963,28.00
7,Philipp Sander,3.866962,4.00
8,Kacper Urbanski,5.542858,6.00
9,Jordan Veretout,2.371675,4.00


In [29]:
comp_leftovers_train.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
428,Bruno Guimaraes,117.460520,80.00
500,Enzo Fernandez,100.308833,75.00
525,Mateus Fernandes,87.168301,15.00
386,James Maddison,85.545442,42.00
215,Scott McTominay,77.504186,50.00
275,Bruno Fernandes,71.439559,50.00
115,Elliot Anderson,71.210364,32.00
113,Xavi Simons,65.867503,70.00
167,Youri Tielemans,65.249331,38.00
268,Ryan Gravenberch,60.299126,75.00


<b>Модель 2</b>

In [6]:
X_train = X_train[['G+A', 'Ast.1', 'Gls.1', 'SoT/90', '90s', 'Fld', 'CrdY', 'SoT%', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G+A', 'Ast.1', 'Gls.1', 'SoT/90', '90s', 'Fld', 'CrdY', 'SoT%', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [13]:
coeffs_df

,coeff
league_GB1,0.578777
90s,0.498539
Age,-0.434598
league_IT1,0.383718
league_ES1,0.304167
league_L1,0.293895
league_FR1,0.253822
Ast.1,0.181157
G+A,0.160573
Gls.1,0.132066


In [14]:
metrics

,MAE,RMSE,MAPE,R2
test,5.768209,14.555255,0.964026,0.531295
train,5.315549,10.920498,0.865460,0.477988


In [15]:
comp_leftovers_test

,Player,Value_pred,Value
0,Sandi Lovric,11.146894,8.500
1,Danila Godiaev,0.937110,1.500
2,Inigo Ruiz de Galarreta,3.063885,3.000
3,Johnny Cardoso,11.521856,25.000
4,Unai Lopez,4.110745,2.500
5,Florian Grillitsch,2.478522,2.000
6,Mahmoud Dahoud,3.045875,3.500
7,Teji Savanier,8.477398,2.500
8,Djaoui Cisse,5.716981,8.000
9,Dean Huiberts,0.869625,0.300


In [17]:
comp_leftovers_train.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
428,Bruno Guimaraes,111.387049,80.00
500,Enzo Fernandez,100.260927,75.00
275,Bruno Fernandes,86.106958,50.00
386,James Maddison,76.885056,42.00
525,Mateus Fernandes,75.470807,15.00
113,Xavi Simons,63.760017,70.00
115,Elliot Anderson,63.654943,32.00
167,Youri Tielemans,62.482759,38.00
268,Ryan Gravenberch,58.025353,75.00
398,Martin Ødegaard,51.738319,85.00


In [18]:
comp_leftovers_train[comp_leftovers_train['Player'].str.contains('Spert')]

,Player,Value_pred,Value
131,Eduard Spertsyan,14.421242,25.0


<b>Модель 3</b>

In [28]:
X_train = X_train[['G+A', 'Ast.1', 'Gls.1', 'SoT/90', '90s', 'Fld', 'CrdY', 'SoT%', 'Age', 'Crs', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G+A', 'Ast.1', 'Gls.1', 'SoT/90', '90s', 'Fld', 'CrdY', 'SoT%', 'Age', 'Crs', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [34]:
coeffs_df

,coeff
league_GB1,0.573213
90s,0.508289
Age,-0.430135
league_IT1,0.374206
league_ES1,0.296953
league_L1,0.288395
league_FR1,0.242921
G+A,0.188811
Ast.1,0.185464
Fld,0.129201


In [35]:
metrics

,MAE,RMSE,MAPE,R2
test,5.686543,13.906516,0.951272,0.572145
train,5.297151,10.922813,0.865137,0.477767


In [36]:
comp_leftovers_test

,Player,Value_pred,Value
0,Sandi Lovric,10.152456,8.500
1,Danila Godiaev,0.933929,1.500
2,Inigo Ruiz de Galarreta,3.106548,3.000
3,Johnny Cardoso,12.092378,25.000
4,Unai Lopez,4.083674,2.500
5,Florian Grillitsch,2.502674,2.000
6,Mahmoud Dahoud,3.088466,3.500
7,Teji Savanier,6.435096,2.500
8,Djaoui Cisse,5.743165,8.000
9,Dean Huiberts,0.890311,0.300


In [37]:
comp_leftovers_train.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
428,Bruno Guimaraes,123.357156,80.00
500,Enzo Fernandez,94.136497,75.00
386,James Maddison,74.287936,42.00
275,Bruno Fernandes,73.404035,50.00
525,Mateus Fernandes,66.934785,15.00
113,Xavi Simons,63.403685,70.00
268,Ryan Gravenberch,62.601550,75.00
115,Elliot Anderson,60.210230,32.00
167,Youri Tielemans,57.210814,38.00
215,Scott McTominay,57.090623,50.00


In [43]:
comp_leftovers_test[comp_leftovers_test['Player'].str.contains('Barco')]

,Player,Value_pred,Value
94,Esequiel Barco,17.286348,14.0


<b>Модель 4</b>

In [47]:
X_train = X_train[['G+A', 'Ast.1', 'Gls.1', 'SoT/90', '90s', 'Fld', 'CrdY', 'SoT%', 'Age', 'Crs', 'Int', 'PK', 'Sh', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G+A', 'Ast.1', 'Gls.1', 'SoT/90', '90s', 'Fld', 'CrdY', 'SoT%', 'Age', 'Crs', 'Int', 'PK', 'Sh', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [53]:
coeffs_df

,coeff
league_GB1,0.586112
90s,0.437282
Age,-0.427508
league_IT1,0.395787
league_ES1,0.316072
league_L1,0.296574
league_FR1,0.254164
G+A,0.206722
Ast.1,0.180166
Fld,0.127575


In [54]:
metrics

,MAE,RMSE,MAPE,R2
test,5.463742,13.214845,0.934374,0.613647
train,5.297484,10.887309,0.862377,0.481157


In [55]:
comp_leftovers_test

,Player,Value_pred,Value
0,Sandi Lovric,10.792036,8.500
1,Danila Godiaev,0.907757,1.500
2,Inigo Ruiz de Galarreta,3.088624,3.000
3,Johnny Cardoso,13.764298,25.000
4,Unai Lopez,4.232315,2.500
5,Florian Grillitsch,2.509089,2.000
6,Mahmoud Dahoud,3.048655,3.500
7,Teji Savanier,5.963070,2.500
8,Djaoui Cisse,5.725256,8.000
9,Dean Huiberts,0.871485,0.300


In [56]:
comp_leftovers_train.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
428,Bruno Guimaraes,121.280403,80.00
500,Enzo Fernandez,92.873187,75.00
386,James Maddison,78.434430,42.00
275,Bruno Fernandes,74.219080,50.00
525,Mateus Fernandes,66.532506,15.00
113,Xavi Simons,66.168543,70.00
268,Ryan Gravenberch,65.459555,75.00
215,Scott McTominay,62.765315,50.00
115,Elliot Anderson,61.395997,32.00
167,Youri Tielemans,57.263853,38.00


In [64]:
comp_leftovers_train[comp_leftovers_train['Player'].str.contains('Sperts')]

,Player,Value_pred,Value
131,Eduard Spertsyan,11.845871,25.0


<b>Модель 5</b>

In [68]:
X_train = X_train[['G+A', 'Gls.1', 'SoT/90', '90s', 'Fld', 'CrdY', 'SoT%', 'Age', 'Crs', 'Int', 'PK', 'Sh', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G+A', 'Gls.1', 'SoT/90', '90s', 'Fld', 'CrdY', 'SoT%', 'Age', 'Crs', 'Int', 'PK', 'Sh', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [74]:
coeffs_df

,coeff
league_GB1,0.578256
Age,-0.410501
G+A,0.408076
90s,0.393690
league_IT1,0.389110
league_ES1,0.306499
league_L1,0.279485
league_FR1,0.252628
Fld,0.129643
CrdY,-0.118318


In [75]:
metrics

,MAE,RMSE,MAPE,R2
test,5.400191,12.703128,0.931033,0.642989
train,5.503649,11.288079,0.877465,0.442255


In [78]:
comp_leftovers_test.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
18,Morgan Gibbs-White,100.370672,50.000
83,Jude Bellingham,91.385108,180.000
123,Jamal Musiala,77.077862,140.000
87,Justin Kluivert,71.856399,35.000
77,Dominik Szoboszlai,70.933475,80.000
13,Sandro Tonali,47.551740,60.000
99,Angelo Stiller,24.043982,45.000
10,Khephren Thuram,21.247696,40.000
111,Guus Til,20.320732,12.000
66,Lamine Camara,18.861745,22.000


In [77]:
comp_leftovers_train.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
428,Bruno Guimaraes,131.394598,80.00
500,Enzo Fernandez,105.726864,75.00
386,James Maddison,89.343911,42.00
215,Scott McTominay,80.451181,50.00
113,Xavi Simons,78.386215,70.00
275,Bruno Fernandes,73.449971,50.00
120,Andrey Santos,69.345831,35.00
498,Sem Steijn,66.753475,16.00
525,Mateus Fernandes,64.496955,15.00
268,Ryan Gravenberch,62.020089,75.00


<b>Модель 6</b>

In [82]:
X_train = X_train[['G+A', 'Gls.1', 'SoT/90', 'Fld', 'CrdY', 'SoT%', 'Age', 'Crs', 'Int', 'PK', 'Sh', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G+A', 'Gls.1', 'SoT/90', 'Fld', 'CrdY', 'SoT%', 'Age', 'Crs', 'Int', 'PK', 'Sh', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [88]:
coeffs_df

,coeff
league_GB1,0.626875
league_IT1,0.459380
G+A,0.452417
Age,-0.399405
league_ES1,0.363291
league_L1,0.324627
league_FR1,0.293720
Int,0.224991
Fld,0.187842
Sh,0.125448


In [89]:
metrics

,MAE,RMSE,MAPE,R2
test,5.254602,11.755020,0.926606,0.694292
train,5.806443,12.176112,0.903290,0.351048


In [90]:
comp_leftovers_test.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
83,Jude Bellingham,123.442940,180.000
18,Morgan Gibbs-White,117.019452,50.000
123,Jamal Musiala,87.428753,140.000
77,Dominik Szoboszlai,70.036507,80.000
87,Justin Kluivert,66.602577,35.000
13,Sandro Tonali,47.853692,60.000
66,Lamine Camara,24.302528,22.000
40,Nadiem Amiri,21.855721,20.000
99,Angelo Stiller,20.553283,45.000
111,Guus Til,20.345750,12.000


In [91]:
comp_leftovers_train.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
428,Bruno Guimaraes,139.582200,80.00
386,James Maddison,115.575742,42.00
275,Bruno Fernandes,105.349084,50.00
500,Enzo Fernandez,100.791502,75.00
215,Scott McTominay,96.241846,50.00
113,Xavi Simons,91.857385,70.00
120,Andrey Santos,75.552536,35.00
232,Carlos Baleba,68.561828,40.00
115,Elliot Anderson,67.098066,32.00
60,Tijjani Reijnders,63.581583,60.00


In [97]:
comp_leftovers_test[comp_leftovers_test['Player'].str.contains('Okish')]

,Player,Value_pred,Value
120,Viktor Okishor,2.785161,0.5
